# CareLoop Clinics — Part 4: GenAI No-Show Analysis

This notebook completes Tasks 4.1–4.4 using the two supplied CSV files.

**Before running:** upload `followup_notes.csv` and `appointments_clean.csv` to the Colab session. Create a fresh Gemini API key in Google AI Studio and enter it when prompted. The key previously pasted into chat is exposed; revoke it and do not reuse it.

Run cells from top to bottom. The notebook creates `notes_analysed.csv` and `README.md`.

In [ ]:
!pip install -q google-genai pydantic pandas

In [ ]:
import json
import time
import pandas as pd
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError

API_KEY = input("Paste your NEW Gemini API key: ")
client = genai.Client(api_key=API_KEY)

MODEL_NAME = "gemini-2.5-flash"

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}

SYSTEM_TEXT = (
    "You are a healthcare appointment follow-up analyst. "
    "Analyse patient call-centre notes carefully. "
    "Return only the requested JSON response. Do not invent information."
)

ALLOWED_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

api_calls = 0
last_raw_response = ""

## Task 4.1 — One validated Gemini call

In [ ]:
def analyse_note(note_text):
    global api_calls, last_raw_response

    if api_calls > 0:
        time.sleep(13)

    prompt = f"""
    Context: A clinic wants to understand why patients missed appointments.
    Role: You are a patient follow-up analyst.
    Action: Read the note and identify the main reason for missing the appointment.
    Allowed root_cause values: {ALLOWED_CAUSES}
    If the reason is unclear, choose other instead of guessing.
    Do not invent any reason.
    Tone: Be accurate, neutral and professional.
    Format: Return JSON with exactly four fields:
    root_cause, confidence, summary, recommended_action.
    Summary must contain at most 25 words. Confidence must be between 0 and 1.

    Patient note:
    {note_text}
    """

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_level="minimal")
        )
    )

    api_calls += 1
    last_raw_response = response.text
    return json.loads(response.text)


notes = pd.read_csv("followup_notes.csv")
N001_note = notes.loc[notes["note_id"] == "N001", "note_text"].iloc[0]
result = analyse_note(str(N001_note).strip())
print(result)

## Task 4.2 — Validate and process the first 10 notes

In [ ]:
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)


results = []

for index, row in notes.head(10).iterrows():
    note_text = row["note_text"]

    if pd.isna(note_text) or str(note_text).strip() == "":
        result = {
            "note_id": row["note_id"],
            "appointment_id": row["appointment_id"],
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank",
            "raw_response": ""
        }
    else:
        raw_response = ""
        reply = {}

        try:
            data = analyse_note(str(note_text).strip())
            raw_response = last_raw_response
            reply = json.loads(raw_response)

            insight = FollowupInsight(**data)

            if insight.root_cause not in ALLOWED_CAUSES:
                raise ValueError("Invalid root_cause value")

            validation_status = "valid"

        except Exception as e:
            print(row["note_id"], "Error:", e)
            validation_status = "invalid"

            # Keep the original reply exactly; copy only fields actually present.
            if raw_response:
                try:
                    reply = json.loads(raw_response)
                except Exception:
                    reply = {}

        result = {
            "note_id": row["note_id"],
            "appointment_id": row["appointment_id"],
            "root_cause": reply.get("root_cause", ""),
            "confidence": reply.get("confidence", ""),
            "summary": reply.get("summary", ""),
            "recommended_action": reply.get("recommended_action", ""),
            "validation_status": validation_status,
            "raw_response": raw_response
        }

    results.append(result)

notes_analysed = pd.DataFrame(results, columns=[
    "note_id", "appointment_id", "root_cause", "confidence",
    "summary", "recommended_action", "validation_status", "raw_response"
])

notes_analysed.to_csv("notes_analysed.csv", index=False)

print(notes_analysed)
print("\nTotal rows:", len(notes_analysed))
print("Total columns:", len(notes_analysed.columns))
print("\nValidation status:")
print(notes_analysed["validation_status"].value_counts())

## Task 4.3 — Test validation and compare prompts

In [ ]:
try:
    bad_reply = FollowupInsight(
        root_cause="other",
        confidence=1.7,
        summary="Patient missed appointment",
        recommended_action="Re-call patient"
    )
except ValidationError as e:
    print("Error 1:")
    print(e)


try:
    bad_reply = FollowupInsight(
        root_cause="other",
        confidence=0.5,
        summary="Patient missed appointment"
    )
except ValidationError as e:
    print("\nError 2:")
    print(e)

In [ ]:
comparison = []

test_notes = notes[notes["note_id"].isin(["N007", "N009"])]

for index, row in test_notes.iterrows():
    note_text = str(row["note_text"]).strip()

    # V1: simple prompt
    if api_calls > 0:
        time.sleep(13)

    response_v1 = client.models.generate_content(
        model=MODEL_NAME,
        contents="Classify this note: " + note_text,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_level="minimal")
        )
    )
    api_calls += 1

    try:
        data_v1 = json.loads(response_v1.text)
        cause_v1 = data_v1.get("root_cause", "")
    except Exception:
        cause_v1 = ""

    comparison.append({
        "note_id": row["note_id"],
        "prompt": "V1",
        "root_cause": cause_v1,
        "allowed": cause_v1 in ALLOWED_CAUSES
    })

    # V2: reuse the CRAFT prompt response already generated in Task 4.2
    v2_row = notes_analysed[
        notes_analysed["note_id"] == row["note_id"]
    ].iloc[0]

    try:
        data_v2 = json.loads(v2_row["raw_response"])
        cause_v2 = data_v2.get("root_cause", "")
    except Exception:
        cause_v2 = ""

    comparison.append({
        "note_id": row["note_id"],
        "prompt": "V2",
        "root_cause": cause_v2,
        "allowed": cause_v2 in ALLOWED_CAUSES
    })

comparison_df = pd.DataFrame(comparison)
print(comparison_df)
print("\nDeploy V2: the CRAFT prompt gives the allowed categories and says to use other rather than guess.")
print("\nTotal API calls used:", api_calls)

## Task 4.4 — Merge, cross-tabulate and create README

In [ ]:
appointments_clean = pd.read_csv("appointments_clean.csv")

merged = notes_analysed.merge(
    appointments_clean[["appointment_id", "reminder_sent"]],
    on="appointment_id",
    how="left"
)

print("Merged data:")
print(merged)
print("\nMerged rows:", len(merged))

cross_table = pd.crosstab(
    merged["root_cause"],
    merged["reminder_sent"]
)
print("\nRoot cause vs Reminder sent:")
print(cross_table)

no_reminder_count = (merged["reminder_sent"] == "No").sum()
forgot_notes = merged[merged["root_cause"] == "forgot_or_no_reminder"]
forgot_total = len(forgot_notes)
forgot_no_reminder = (forgot_notes["reminder_sent"] == "No").sum()

total_appointments = len(appointments_clean)
total_no_shows = (appointments_clean["no_show"] == 1).sum()
no_show_rate = round(total_no_shows / total_appointments * 100, 1)
total_clinics = appointments_clean["clinic_id"].nunique()
total_notes = len(notes_analysed)

print("\nAppointments:", total_appointments)
print("No-shows:", total_no_shows)
print("No-show rate (%):", no_show_rate)
print("Clinics:", total_clinics)
print("Notes analysed:", total_notes)
print("No reminders in 10 notes:", no_reminder_count)
print("Forgot/no-reminder notes:", forgot_total)
print("Forgot/no-reminder with reminder_sent=No:", forgot_no_reminder)

### Enter your actual Part 1–3 results

The source files provided here do not contain your earlier Part 1–3 report outputs or model evaluation. Enter those values from your own completed notebook; do not use invented AUC, threshold, or findings.

In [ ]:
part1_finding = input("Part 1 finding (one concise sentence): ")
part2_finding = input("Part 2 finding (one concise sentence): ")
model_name = input("Part 3 selected model name: ")
auc_score = input("Part 3 AUC score: ")
threshold = input("Part 3 chosen threshold: ")

github_link = input("Public GitHub repository URL: ")

In [ ]:
readme = f\"\"\"# CareLoop Clinics — No-Show Guard

**GitHub Repository:** {github_link}

## Project Summary

Part 1: {part1_finding}

Part 2: {part2_finding}

Part 3: {model_name} achieved an AUC of {auc_score} at a threshold of {threshold}.

Part 4: Analysed {total_notes} patient notes; {no_reminder_count} of 10 appointments had reminder_sent = No, and {forgot_no_reminder} of {forgot_total} forgotten-reminder cases had reminder_sent = No.

## Business Recommendation

The appointments dataset contains {total_appointments} appointments, {total_no_shows} no-shows, a {no_show_rate}% no-show rate, and {total_clinics} clinics. {part1_finding} {part2_finding} The selected {model_name} achieved an AUC of {auc_score} at a threshold of {threshold}. In the 10-note illustration, {no_reminder_count} appointments had no reminder recorded; {forgot_no_reminder} of {forgot_total} forgotten-reminder cases had no reminder recorded. The clinic should use the selected model and chosen threshold to identify higher-risk appointments, then prioritise reminder messages and follow-up calls within available call capacity. It should also review transport, fee, illness and work-schedule reasons before choosing operational interventions. These note results are illustrative and should not be treated as clinic-wide estimates.

## Limitations

1. The 10 notes are an illustration and cannot show the real reasons clinic-wide.
2. The data are observational, so associations are not causes.
3. A threshold that really depends on call capacity.

## Tools and Technologies

- Python and Pandas
- Google Gemini Flash API
- Pydantic
- Google Colab
\"\"\"

with open("README.md", "w", encoding="utf-8") as file:
    file.write(readme)

print("README.md created successfully.")
print("\nFiles created: notes_analysed.csv and README.md")

## GitHub submission checklist

1. Create a **Public** repository.
2. Upload this notebook, `followup_notes.csv`, `appointments_clean.csv`, `notes_analysed.csv`, and `README.md`, plus the other files required under your assignment's Submission section.
3. Search the notebook and repository for `AQ.` and `AIza` to ensure no API key is present.
4. Never paste the API key into a code cell or commit it.